*(Optional / advanced.)* In 4-4 we used FinBERT models that the paper's authors already fine-tuned for us. Here we fine-tune the pretrained FinBERT ourselves, on a small labeled dataset, following the same recipe as the FinBERT repository's own `finetune.ipynb` (learning rate 2e-5, a handful of epochs).

You would fine-tune your own model when your task doesn't match an existing FinBERT variant -- a custom label taxonomy, a different text domain, or a research design that calls for a model trained only on your own sample.

Today's data is the **Financial PhraseBank** (Malo et al. 2014) -- ~4,800 sentences from financial news, hand-labeled positive/neutral/negative by 16 annotators. It's public, small, and it's the same robustness-check dataset the FinBERT paper itself uses (footnote 11). We saved the highest-agreement subset to `data/financial_phrasebank.csv` (see `data/financial_phrasebank_README.txt` for the citation and license).

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer, set_seed
from datasets import Dataset

set_seed(42)  # the classification head is randomly initialized -- seed it so results are reproducible

### 1. A deliberately small training sample

The paper's headline small-sample result (Table 2): FinBERT trained on just **10% of its training data still reaches 81.3% accuracy** -- more than the best non-BERT model achieves with the *full* training sample. We replicate the same idea here: instead of using all ~2,260 labeled sentences, we sample only 400 (300 train / 50 validation / 50 test) and see how far that gets us.

In [ ]:
df = pd.read_csv('data/financial_phrasebank.csv')
label2id = {'neutral': 0, 'positive': 1, 'negative': 2}
id2label = {v: k for k, v in label2id.items()}
df['label_id'] = df['label'].map(label2id)

sample, _ = train_test_split(df, train_size=400, stratify=df['label_id'], random_state=42)
train_df, temp_df = train_test_split(sample, train_size=300, stratify=sample['label_id'], random_state=42)
val_df, test_df = train_test_split(temp_df, train_size=50, stratify=temp_df['label_id'], random_state=42)

print('train / val / test:', len(train_df), len(val_df), len(test_df))
train_df['label'].value_counts()

### 2. Tokenize and wrap as a `datasets.Dataset`

`Trainer` (from the `transformers` library) expects a Hugging Face `Dataset`, not a plain DataFrame.

In [ ]:
tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-pretrain')

def tokenize(batch):
    return tokenizer(batch['sentence'], truncation=True, padding='max_length', max_length=64)

def to_dataset(frame: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(frame[['sentence', 'label_id']].rename(columns={'label_id': 'label'}), preserve_index=False)
    return ds.map(tokenize, batched=True)

train_ds = to_dataset(train_df)
val_ds = to_dataset(val_df)
test_ds = to_dataset(test_df)

### 3. Fine-tune

We start from `yiyanghkust/finbert-pretrain` -- FinBERT *before* any task-specific fine-tuning -- and add a fresh 3-way classification head on top (the "MISSING: classifier.weight / classifier.bias" message below is expected: that head has never been trained yet, which is exactly what we're about to do). This mirrors the FinBERT repository's own `finetune.ipynb`, scaled down to run quickly on a laptop CPU.

In [ ]:
model = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-pretrain', num_labels=3)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {'accuracy': accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir='finbert_finetune_demo',  # local checkpoints only -- do not commit this folder
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy='epoch',
    logging_strategy='epoch',
    save_strategy='no',
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

On a laptop CPU, this takes roughly a minute for 300 sentences / 3 epochs -- fine-tuning is fast precisely *because* pretraining already did the expensive part.

In [ ]:
test_metrics = trainer.evaluate(test_ds)
print(f"Accuracy on the held-out test set (n={len(test_df)}): {test_metrics['eval_accuracy']:.1%}")

### 4. How much did fine-tuning actually buy us?

For comparison, let's evaluate the *already fine-tuned* `finbert-tone` model (from 4-4, trained by the paper's authors on analyst-report sentences -- a different dataset entirely) on this exact same test set, with no further training at all.

In [ ]:
zero_shot_tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-tone')
zero_shot_model = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-tone', num_labels=3)
zero_shot_model.eval()

with torch.no_grad():
    inputs = zero_shot_tokenizer(test_df['sentence'].tolist(), return_tensors='pt', padding=True, truncation=True, max_length=64)
    zero_shot_preds = torch.argmax(zero_shot_model(**inputs)[0], dim=1).tolist()

zero_shot_acc = accuracy_score(test_df['label_id'].tolist(), zero_shot_preds)
print(f'Zero-shot finbert-tone accuracy on the same test set: {zero_shot_acc:.1%}')
print(f"Our 300-sentence fine-tune's accuracy:                 {test_metrics['eval_accuracy']:.1%}")

Both numbers should be reasonably high (typically 80-95%, depending on the random split and initialization). `finbert-tone` was never trained on a single Financial PhraseBank sentence, yet it already generalizes well to it -- a direct illustration of what pretraining on 4.9 billion words of financial text buys you (Section 3 of the paper). And in about a minute of CPU time, fine-tuning on just 300 task-specific sentences gets our own model to a comparable accuracy, without any access to the paper's original training data.

This is the practical takeaway of the paper's small-sample-size analysis: you very rarely need thousands of hand-labeled examples to get a domain-adapted LLM to a useful accuracy -- which matters, because labeling is usually the most expensive part of a supervised-learning research design. (If you re-run this notebook, don't be surprised if the exact numbers shift by a few points -- with only 50 test sentences, each one is worth 2 percentage points.)

If you wanted to reuse this fine-tuned model later, `trainer.save_model('finbert_finetune_demo')` saves it to disk -- but treat that folder as local scratch output, not something to check into the course repository (a fine-tuned BERT checkpoint is several hundred MB).